# GPU Random Forest Collapsed Feature-Family MTL

Train one CUDA-accelerated random-forest classifier per FMP feature family using a collapsed target label.

Shape:

- Universe: FMP common equities with `market_cap >= 1_000_000_000_000`.
- Features: numeric Quant Warehouse feature families, one model per feature family.
- Targets: one collapsed multiclass label instead of one model per event task.
- Event labels: bullish event rows only (`congress_buy`, `insider_buy`, `analyst_upgrade`, `price_target_raise`, `earnings_beat`). Mirror sell/down/cut/miss rows are not treated as bearish training labels here.
- Oracle labels: `YE: [1..12]` long/short entries from the optimal-trade solver, collapsed into `oracle_long` and `oracle_short` classes.

This is the random-forest analogue of the Flair MTL notebook: transformers collapse feature-family text into one encoder, while this notebook collapses target tasks into one label so tabular RF models do not explode into hundreds of family/task combinations.


In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import random
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# Prefer the local quant-warehouse checkout while this notebook and the helper code are evolving together.
QUANT_WAREHOUSE_ROOT = Path('/home/jlee153232/PycharmProjects/quant-warehouse')
if str(QUANT_WAREHOUSE_ROOT) not in sys.path:
    sys.path.insert(0, str(QUANT_WAREHOUSE_ROOT))

import cupy as cp
import cudf
from cuml.ensemble import RandomForestClassifier as CuRandomForestClassifier

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_collapsed_bullish_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
warnings.filterwarnings('ignore', category=FutureWarning)

print('cupy', cp.__version__, 'cuda_devices', cp.cuda.runtime.getDeviceCount())
print('cudf', cudf.__version__)


cupy 14.1.1 cuda_devices 1
cudf 26.06.00


In [2]:
RANDOM_SEED = 20260702
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '2018-01-01'
END_DATE = None
TRAIN_END = pd.Timestamp('2023-12-31')
DEV_END = pd.Timestamp('2024-12-31')

MIN_FEATURE_COVERAGE = 0.50
MAX_FEATURES_PER_FAMILY = 50
MIN_ROWS_PER_FAMILY = 500
MIN_CLASSES_PER_FAMILY = 2

RF_PARAMS = {
    'n_estimators': 300,
    'max_depth': 16,
    'max_features': 'sqrt',
    'n_bins': 128,
    'random_state': RANDOM_SEED,
    'n_streams': 8,
}

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    max_features_per_family=MAX_FEATURES_PER_FAMILY,
)
TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=START_DATE,
    end_date=END_DATE,
    event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
FEATURE_CONFIG, TARGET_CONFIG


(FamilyEvaluationConfig(provider='fmp', market_cap_min=1000000000000, country='US', exchanges=('NASDAQ', 'NYSE', 'AMEX'), screen_limit=5000, start_date='2018-01-01', end_date=None, filing_lag_days=45, horizons=(20, 60, 120), min_observations=120, max_features_per_family=50),
 BinaryTargetConfig(provider='fmp', start_date='2018-01-01', end_date=None, event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'), oracle_trade_k_by_frequency={'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}, oracle_trade_min_profit_pct=0.01, oracle_trade_long_entry_price_col='high', oracle_trade_long_exit_price_col='low', oracle_trade_short_entry_price_col='low', oracle_trade_short_exit_price_col='high', event_alignment_tolerance_days=7, collapsed_bullish_event_types=('congress_buy', 'insider_buy', 'analyst_upgrade', 'price_target_raise', 'earnings_beat')))

## Build 1T+ Feature and Target Panels

The label construction is event-first. No no-event dates are materialized as negative examples. Event mirror rows are excluded from the collapsed bullish event class because the current EDA does not support treating sells/down/cuts/misses as simple bearish labels.


In [3]:
started = perf_counter()
WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(backend=WAREHOUSE.backend, catalog=WAREHOUSE.catalog)

symbols, raw_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)
print({'universe_source': universe_source, 'eligible_symbols': len(symbols)})
display(universe_eligibility.loc[universe_eligibility['eligible']].head(25))


{'universe_source': 'openbb:fmp', 'eligible_symbols': 14}


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)
selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel,
    raw_feature_metadata,
    max_features=MAX_FEATURES_PER_FAMILY,
)
feature_panel = raw_feature_panel[['symbol', 'date', *selected_features]].copy()
print({
    'raw_feature_panel_rows': len(raw_feature_panel),
    'selected_feature_columns': len(selected_features),
    'selected_feature_families': selected_feature_metadata['family'].nunique(),
    **feature_timings,
})
display(
    selected_feature_metadata.groupby(['source', 'family'], as_index=False)
    .agg(feature_count=('feature', 'nunique'))
    .sort_values(['source', 'family'])
)


{'raw_feature_panel_rows': 27701, 'selected_feature_columns': 365, 'selected_feature_families': 15, 'raw_panel_build_seconds': 1.2861643726937473}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,39


In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)
event_symbols = tuple(event_diagnostics.loc[event_diagnostics['combined_rows'].gt(0), 'symbol'].sort_values())
feature_panel = feature_panel.loc[feature_panel['symbol'].isin(event_symbols)].copy()

collapsed_event_panel, collapsed_event_metadata = build_collapsed_bullish_event_target_panel(
    feature_panel[['symbol', 'date']],
    events,
    TARGET_CONFIG,
)
oracle_panel, oracle_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)
print({
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'event_load_seconds': round(event_load_seconds, 3),
    'collapsed_event_rows': len(collapsed_event_panel),
    'oracle_rows': len(oracle_panel),
    'oracle_columns': len(oracle_metadata),
    'oracle_seconds': round(oracle_seconds, 3),
})
display(event_diagnostics.sort_values('combined_rows', ascending=False).head(20))


{'event_symbols': 13, 'event_rows': 5179, 'event_load_seconds': 0.88, 'collapsed_event_rows': 27690, 'oracle_rows': 27690, 'oracle_columns': 36, 'oracle_seconds': 1.664}


,symbol,cached_rows,historical_rows,combined_rows,event_families
9,MSFT,34,945,979,"(analyst_rating, congress, earnings, insider, ..."
0,AAPL,32,820,852,"(analyst_rating, congress, earnings, insider, ..."
11,NVDA,25,799,824,"(analyst_rating, congress, earnings, insider, ..."
1,AMZN,34,719,753,"(analyst_rating, congress, earnings, insider, ..."
6,GOOGL,34,581,615,"(analyst_rating, congress, earnings, insider, ..."
13,TSLA,32,456,488,"(analyst_rating, congress, earnings, insider, ..."
8,META,34,437,471,"(analyst_rating, congress, earnings, insider, ..."
3,BRK-A,0,34,34,"(earnings,)"
4,BRK-B,0,34,34,"(earnings,)"
5,GOOG,0,34,34,"(earnings,)"


## Collapse Targets to One Label Column

The final random-forest label is multiclass:

- `event_bullish`: configured bullish event rows only.
- `oracle_long`: any `YE k=1..12` oracle long entry where no oracle short also fires on the same row.
- `oracle_short`: any `YE k=1..12` oracle short entry where no oracle long also fires on the same row.

Rows with both oracle long and short after collapsing across `k` are ambiguous for a single-label classifier and are dropped.


In [6]:
EVENT_TARGET = 'target_event_collapsed__bullish'
EVENT_ACTIVITY = f'_target_activity__{EVENT_TARGET}'


def collapsed_label_rows(collapsed_event_panel: pd.DataFrame, oracle_panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = []
    diagnostics = []

    if EVENT_TARGET in collapsed_event_panel.columns:
        event_frame = collapsed_event_panel.copy()
        event_frame[EVENT_ACTIVITY] = pd.to_numeric(event_frame.get(EVENT_ACTIVITY, 0), errors='coerce').fillna(0).astype('int8')
        event_frame[EVENT_TARGET] = pd.to_numeric(event_frame[EVENT_TARGET], errors='coerce').fillna(0).astype('int8')
        bullish = event_frame.loc[event_frame[EVENT_TARGET].eq(1), ['symbol', 'date']].copy()
        bullish['collapsed_label'] = 'event_bullish'
        bullish['label_source'] = 'event_collapsed'
        frames.append(bullish)
        diagnostics.append({
            'source': 'event_collapsed',
            'candidate_rows': int(event_frame[EVENT_ACTIVITY].gt(0).sum()),
            'used_rows': len(bullish),
            'dropped_rows': int(event_frame[EVENT_ACTIVITY].gt(0).sum()) - len(bullish),
            'note': 'mirror/non-bullish event rows excluded',
        })

    long_cols = sorted(column for column in oracle_panel.columns if column.startswith('target_oracle_trade_entry__') and column.endswith('_long'))
    short_cols = sorted(column for column in oracle_panel.columns if column.startswith('target_oracle_trade_entry__') and column.endswith('_short'))
    if long_cols and short_cols:
        oracle = oracle_panel[['symbol', 'date', *long_cols, *short_cols]].copy()
        long_any = oracle[long_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        short_any = oracle[short_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        ambiguous = long_any & short_any
        long_rows = oracle.loc[long_any & ~short_any, ['symbol', 'date']].copy()
        long_rows['collapsed_label'] = 'oracle_long'
        long_rows['label_source'] = 'oracle_trade'
        short_rows = oracle.loc[short_any & ~long_any, ['symbol', 'date']].copy()
        short_rows['collapsed_label'] = 'oracle_short'
        short_rows['label_source'] = 'oracle_trade'
        frames.extend([long_rows, short_rows])
        diagnostics.append({
            'source': 'oracle_trade',
            'candidate_rows': int((long_any | short_any).sum()),
            'used_rows': len(long_rows) + len(short_rows),
            'dropped_rows': int(ambiguous.sum()),
            'note': 'ambiguous long+short rows dropped after k collapse',
        })

    if not frames:
        return pd.DataFrame(columns=['symbol', 'date', 'collapsed_label', 'label_source']), pd.DataFrame(diagnostics)
    labels = pd.concat(frames, ignore_index=True)
    labels['symbol'] = labels['symbol'].astype(str).str.upper()
    labels['date'] = pd.to_datetime(labels['date'], errors='coerce').dt.normalize()
    labels = labels.dropna(subset=['symbol', 'date', 'collapsed_label']).drop_duplicates()
    return labels.sort_values(['date', 'symbol', 'collapsed_label']).reset_index(drop=True), pd.DataFrame(diagnostics)

label_rows, label_diagnostics = collapsed_label_rows(collapsed_event_panel, oracle_panel)
print({'collapsed_label_rows': len(label_rows), 'classes': sorted(label_rows['collapsed_label'].unique())})
display(label_diagnostics)
display(label_rows.groupby(['label_source', 'collapsed_label'], as_index=False).agg(rows=('symbol', 'size'), symbols=('symbol', 'nunique'), min_date=('date', 'min'), max_date=('date', 'max')))


{'collapsed_label_rows': 4568, 'classes': ['event_bullish', 'oracle_long', 'oracle_short']}


,source,candidate_rows,used_rows,dropped_rows,note
0,event_collapsed,3491,1934,1557,mirror/non-bullish event rows excluded
1,oracle_trade,2634,2634,0,ambiguous long+short rows dropped after k coll...


,label_source,collapsed_label,rows,symbols,min_date,max_date
0,event_collapsed,event_bullish,1934,13,2018-01-25,2026-06-22
1,oracle_trade,oracle_long,1335,13,2018-01-02,2026-06-12
2,oracle_trade,oracle_short,1299,13,2018-01-05,2026-06-22


## Train One CUDA Random Forest Per Feature Family

Each model sees only one feature family. This answers the tabular analogue of the feature-family MTL question: which family can separate the collapsed target labels without training a separate model for every event task?


In [7]:
def prepare_family_dataset(
    feature_panel: pd.DataFrame,
    feature_metadata: pd.DataFrame,
    labels: pd.DataFrame,
    *,
    source: str,
    family: str,
    min_feature_coverage: float,
) -> tuple[pd.DataFrame, list[str]]:
    family_meta = feature_metadata.loc[
        feature_metadata['source'].astype(str).eq(source) & feature_metadata['family'].astype(str).eq(family)
    ]
    features = [feature for feature in family_meta['feature'].drop_duplicates().tolist() if feature in feature_panel.columns]
    numeric_features = [feature for feature in features if pd.to_numeric(feature_panel[feature], errors='coerce').notna().any()]
    if not numeric_features:
        return pd.DataFrame(), []
    merged = labels.merge(feature_panel[['symbol', 'date', *numeric_features]], on=['symbol', 'date'], how='inner')
    if merged.empty:
        return merged, numeric_features
    numeric = merged[numeric_features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    merged = merged.loc[coverage.ge(min_feature_coverage)].copy()
    if merged.empty:
        return merged, numeric_features
    numeric = numeric.loc[merged.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    merged[numeric_features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return merged.reset_index(drop=True), numeric_features


def temporal_split(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    dates = pd.to_datetime(frame['date'], errors='coerce')
    return {
        'train': frame.loc[dates.le(TRAIN_END)].copy(),
        'dev': frame.loc[dates.gt(TRAIN_END) & dates.le(DEV_END)].copy(),
        'test': frame.loc[dates.gt(DEV_END)].copy(),
    }


def fit_gpu_random_forest(train: pd.DataFrame, features: list[str], labels: list[str]) -> tuple[CuRandomForestClassifier, LabelEncoder]:
    encoder = LabelEncoder()
    y = encoder.fit_transform(train['collapsed_label'].astype(str))
    model = CuRandomForestClassifier(**RF_PARAMS)
    X_gpu = cudf.from_pandas(train[features].astype('float32'))
    y_gpu = cudf.Series(y.astype('int32'))
    model.fit(X_gpu, y_gpu)
    return model, encoder


def predict_labels(model: CuRandomForestClassifier, encoder: LabelEncoder, frame: pd.DataFrame, features: list[str]) -> np.ndarray:
    pred = model.predict(cudf.from_pandas(frame[features].astype('float32')))
    if hasattr(pred, 'to_numpy'):
        pred_values = pred.to_numpy()
    else:
        pred_values = cp.asnumpy(pred)
    return encoder.inverse_transform(np.asarray(pred_values).astype(int))


def score_split(model: CuRandomForestClassifier, encoder: LabelEncoder, frame: pd.DataFrame, features: list[str], split: str) -> dict[str, object]:
    if frame.empty:
        return {'split': split, 'rows': 0, 'accuracy': np.nan, 'balanced_accuracy': np.nan, 'macro_f1': np.nan}
    y_true = frame['collapsed_label'].astype(str).to_numpy()
    y_pred = predict_labels(model, encoder, frame, features)
    return {
        'split': split,
        'rows': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }


In [8]:
results = []
reports = {}
models = {}

family_keys = (
    selected_feature_metadata[['source', 'family']]
    .drop_duplicates()
    .sort_values(['source', 'family'])
    .itertuples(index=False, name=None)
)
train_started = perf_counter()
for source, family in family_keys:
    family_frame, features = prepare_family_dataset(
        feature_panel,
        selected_feature_metadata,
        label_rows,
        source=str(source),
        family=str(family),
        min_feature_coverage=MIN_FEATURE_COVERAGE,
    )
    if len(family_frame) < MIN_ROWS_PER_FAMILY or family_frame['collapsed_label'].nunique() < MIN_CLASSES_PER_FAMILY:
        results.append({
            'source': source,
            'family': family,
            'status': 'skipped_sparse',
            'features': len(features),
            'rows': len(family_frame),
            'classes': family_frame['collapsed_label'].nunique() if not family_frame.empty else 0,
        })
        continue
    splits = temporal_split(family_frame)
    if splits['train']['collapsed_label'].nunique() < MIN_CLASSES_PER_FAMILY:
        results.append({
            'source': source,
            'family': family,
            'status': 'skipped_train_class_sparse',
            'features': len(features),
            'rows': len(family_frame),
            'train_rows': len(splits['train']),
            'classes': family_frame['collapsed_label'].nunique(),
        })
        continue
    model_started = perf_counter()
    model, encoder = fit_gpu_random_forest(splits['train'], features, sorted(family_frame['collapsed_label'].unique()))
    elapsed = perf_counter() - model_started
    models[(source, family)] = {'model': model, 'encoder': encoder, 'features': features}
    row = {
        'source': source,
        'family': family,
        'status': 'ok',
        'features': len(features),
        'rows': len(family_frame),
        'classes': family_frame['collapsed_label'].nunique(),
        'train_rows': len(splits['train']),
        'dev_rows': len(splits['dev']),
        'test_rows': len(splits['test']),
        'fit_seconds': elapsed,
    }
    for split_name, split_frame in splits.items():
        split_scores = score_split(model, encoder, split_frame, features, split_name)
        for metric, value in split_scores.items():
            if metric not in {'split'}:
                row[f'{split_name}_{metric}'] = value
    if not splits['test'].empty:
        y_true = splits['test']['collapsed_label'].astype(str).to_numpy()
        y_pred = predict_labels(model, encoder, splits['test'], features)
        reports[(source, family)] = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    results.append(row)

results_df = pd.DataFrame(results).sort_values(['status', 'test_macro_f1', 'dev_macro_f1', 'rows'], ascending=[True, False, False, False]).reset_index(drop=True)
print({'trained_models': len(models), 'elapsed_seconds': round(perf_counter() - train_started, 3)})
display(results_df)


[20:30:31] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:30:33] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:30:35] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:30:36] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:30:38] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


...

[20:30:52] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:30:53] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


{'trained_models': 15, 'elapsed_seconds': 25.781}


[20:30:55] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,source,family,status,features,rows,classes,train_rows,dev_rows,test_rows,fit_seconds,train_accuracy,train_balanced_accuracy,train_macro_f1,dev_accuracy,dev_balanced_accuracy,dev_macro_f1,test_accuracy,test_balanced_accuracy,test_macro_f1
0,financetoolkit,ft_growth_balance,ok,50,4568,3,2892,541,1135,1.8951,0.5249,0.5092,0.4980,0.3900,0.3647,0.3516,0.4881,0.4144,0.4067
1,financetoolkit,ft_ratios_liquidity,ok,7,4568,3,2892,541,1135,1.5661,0.5249,0.5091,0.4978,0.3863,0.3667,0.3653,0.4881,0.4004,0.3927
2,financetoolkit,ft_ratios_efficiency,ok,5,4568,3,2892,541,1135,1.5321,0.5246,0.5087,0.4970,0.4954,0.4449,0.3990,0.4441,0.4085,0.3883
3,financetoolkit,ft_ratios_solvency,ok,15,4568,3,2892,541,1135,1.5170,0.5249,0.5089,0.4970,0.4566,0.4276,0.4251,0.4308,0.3981,0.3861
4,financetoolkit,ft_ratios_profitability,ok,14,4568,3,2892,541,1135,1.5057,0.5249,0.5089,0.4973,0.3993,0.3858,0.3773,0.4449,0.3939,0.3816
5,financetoolkit,ft_growth_income,ok,38,4568,3,2892,541,1135,1.5584,0.5246,0.5086,0.4970,0.3863,0.3709,0.3547,0.4441,0.3900,0.3693
6,fmp,fmp_daily_mcap_yield,ok,14,4568,3,2892,541,1135,1.5703,0.9080,0.9061,0.9104,0.4233,0.4255,0.3930,0.3991,0.3833,0.3627
7,fmp,fmp_daily_mcap_multiple,ok,14,4568,3,2892,541,1135,1.5684,0.9104,0.9085,0.9128,0.4011,0.4125,0.3617,0.3991,0.3942,0.3612
8,financetoolkit,ft_ratios_valuation,ok,24,4568,3,2892,541,1135,1.5643,0.5249,0.5092,0.4979,0.4640,0.4215,0.3864,0.4705,0.3829,0.3546
9,fmp,fmp_daily_ev_yield,ok,7,4568,3,2892,541,1135,1.5962,0.9015,0.9000,0.9035,0.4233,0.4018,0.4006,0.3868,0.3436,0.3407


## Result Inspection

The top table ranks feature families by test macro F1. Use macro F1 and balanced accuracy rather than raw accuracy because the collapsed classes are not guaranteed to be balanced.


In [9]:
ok_results = results_df.loc[results_df['status'].eq('ok')].copy()
metric_cols = ['source', 'family', 'features', 'rows', 'train_rows', 'dev_rows', 'test_rows', 'fit_seconds', 'dev_macro_f1', 'test_macro_f1', 'test_balanced_accuracy', 'test_accuracy']
display(ok_results[[column for column in metric_cols if column in ok_results.columns]].head(30))

if reports:
    best_key = tuple(ok_results.iloc[0][['source', 'family']])
    best_report = pd.DataFrame(reports[best_key]).T
    print({'best_family': best_key})
    display(best_report)


,source,family,features,rows,train_rows,dev_rows,test_rows,fit_seconds,dev_macro_f1,test_macro_f1,test_balanced_accuracy,test_accuracy
0,financetoolkit,ft_growth_balance,50,4568,2892,541,1135,1.8951,0.3516,0.4067,0.4144,0.4881
1,financetoolkit,ft_ratios_liquidity,7,4568,2892,541,1135,1.5661,0.3653,0.3927,0.4004,0.4881
2,financetoolkit,ft_ratios_efficiency,5,4568,2892,541,1135,1.5321,0.3990,0.3883,0.4085,0.4441
3,financetoolkit,ft_ratios_solvency,15,4568,2892,541,1135,1.5170,0.4251,0.3861,0.3981,0.4308
4,financetoolkit,ft_ratios_profitability,14,4568,2892,541,1135,1.5057,0.3773,0.3816,0.3939,0.4449
5,financetoolkit,ft_growth_income,38,4568,2892,541,1135,1.5584,0.3547,0.3693,0.3900,0.4441
6,fmp,fmp_daily_mcap_yield,14,4568,2892,541,1135,1.5703,0.3930,0.3627,0.3833,0.3991
7,fmp,fmp_daily_mcap_multiple,14,4568,2892,541,1135,1.5684,0.3617,0.3612,0.3942,0.3991
8,financetoolkit,ft_ratios_valuation,24,4568,2892,541,1135,1.5643,0.3864,0.3546,0.3829,0.4705
9,fmp,fmp_daily_ev_yield,7,4568,2892,541,1135,1.5962,0.4006,0.3407,0.3436,0.3868


{'best_family': ('financetoolkit', 'ft_growth_balance')}


,precision,recall,f1-score,support
event_bullish,0.6634,0.6442,0.6537,624.0000
oracle_long,0.2689,0.4104,0.3249,251.0000
oracle_short,0.3356,0.1885,0.2414,260.0000
accuracy,0.4881,0.4881,0.4881,0.4881
macro avg,0.4226,0.4144,0.4067,"1,135.0000"
weighted avg,0.5011,0.4881,0.4865,"1,135.0000"


In [10]:
analysis_lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} FMP symbols with market cap >= ${MIN_MARKET_CAP:,.0f}; {len(event_symbols)} had event coverage.',
    f'- Collapsed labels: {len(label_rows):,} rows across {label_rows["collapsed_label"].nunique()} classes: {tuple(sorted(label_rows["collapsed_label"].unique()))}.',
    f'- Oracle config: {TARGET_CONFIG.oracle_trade_k_by_frequency}.',
    f'- Trained GPU random-forest models: {len(models)} feature-family models.',
]
if not ok_results.empty:
    best = ok_results.iloc[0]
    analysis_lines.extend([
        '',
        'Best feature family by test macro F1:',
        f'- {best["source"]}.{best["family"]}: test_macro_f1={best.get("test_macro_f1", np.nan):.4f}, test_balanced_accuracy={best.get("test_balanced_accuracy", np.nan):.4f}, rows={int(best["rows"]):,}, features={int(best["features"])}.',
        '',
        'Interpretation:',
        '- This is a task-collapse experiment, not a trading backtest.',
        '- Event sell/down/cut/miss mirrors are intentionally excluded rather than treated as bearish labels.',
        '- Oracle long/short labels remain because the optimal-trade solver explicitly produces both sides.',
        '- One model per feature family makes RF feasible while preserving the feature-family comparison that the transformer notebooks handled with text encoders.',
    ])
else:
    analysis_lines.extend(['', '- No feature-family model met the minimum row/class thresholds.'])

from IPython.display import Markdown, display
display(Markdown('\n'.join(analysis_lines)))


## Written Analysis

- Universe: 14 FMP symbols with market cap >= $1,000,000,000,000; 13 had event coverage.
- Collapsed labels: 4,568 rows across 3 classes: ('event_bullish', 'oracle_long', 'oracle_short').
- Oracle config: {'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}.
- Trained GPU random-forest models: 15 feature-family models.

Best feature family by test macro F1:
- financetoolkit.ft_growth_balance: test_macro_f1=0.4067, test_balanced_accuracy=0.4144, rows=4,568, features=50.

Interpretation:
- This is a task-collapse experiment, not a trading backtest.
- Event sell/down/cut/miss mirrors are intentionally excluded rather than treated as bearish labels.
- Oracle long/short labels remain because the optimal-trade solver explicitly produces both sides.
- One model per feature family makes RF feasible while preserving the feature-family comparison that the transformer notebooks handled with text encoders.

## Written Analysis

- Universe: 14 FMP symbols with market cap >= $1,000,000,000,000; 13 had event coverage.
- Collapsed labels: 4,568 rows across 3 classes: ('event_bullish', 'oracle_long', 'oracle_short').
- Oracle config: {'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}.
- Trained GPU random-forest models: 15 feature-family models.

Best feature family by test macro F1:
- financetoolkit.ft_growth_balance: test_macro_f1=0.4067, test_balanced_accuracy=0.4144, rows=4,568, features=50.

Interpretation:
- This is a task-collapse experiment, not a trading backtest.
- Event sell/down/cut/miss mirrors are intentionally excluded rather than treated as bearish labels.
- Oracle long/short labels remain because the optimal-trade solver explicitly produces both sides.
- One model per feature family makes RF feasible while preserving the feature-family comparison that the transformer notebooks handled with text encoders.